# Nail segmentation → TensorFlow Lite (Colab outline)

**Goal:** Train a binary nail mask model and export **`nail_seg.tflite`** for Android (`NailSegmentationHelper`).

**Data layout (Kaggle nail dataset):**
```
DATA_ROOT/
  train/images/   *.jpg
  train/masks/    *.png   # 0 = background, 255 = nail
  valid/images/   (or val/images/)
  valid/masks/
  test/images/
  test/masks/
  NailSegmentationV1.csv   # optional metadata — not required for training
```

The notebook prefers **`train` + `valid`** (or **`val`**) splits automatically instead of random **`VAL_SPLIT`**.

**Flow:** paths → `tf.data` → **MobileNetV3-Small U-Net** (skip connections at strides **32 / 16 / 8 / optionally 4**) → Dice+BCE → val IoU → **SavedModel** → **TFLite float32** → verify interpreter shapes.

### Open-source dataset (Kaggle)

This notebook can load **[Nail Segmentation Dataset](https://www.kaggle.com/datasets/muhammadhammad261/nail-segmentation-dataset)** (`muhammadhammad261/nail-segmentation-dataset`). Layout: **`train/`**, **`valid/`** (or **`val/`**), **`test/`**, each with **`images/`** (JPG) and **`masks/`** (PNG, 0/255). **`NailSegmentationV1.csv`** is optional metadata.

> If discovery fails, run `!find DATA_ROOT -type d | head -40` and set paths manually.

## 1. Dependencies

Colab includes TensorFlow; uncomment to pin a version.

In [ ]:
# !pip install -q "tensorflow>=2.13,<2.17"
# !pip install -q kaggle   # only if using Kaggle download (Section 2b)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from typing import Tuple, Optional, List

print("TensorFlow", tf.__version__)

## 2. Config

**`IMG_SIZE`** must match TFLite input (e.g. 256). Android resizes photos to the model input shape.

- **`DATA_ROOT`**: dataset root with **`train/`**, **`valid/`** or **`val/`**, optionally **`test/`**.
- **`USE_OFFICIAL_SPLITS`**: if `True` (default), use `train/images` + `train/masks` and `valid|val/images` + `masks`. If `False`, merge all pairs under `DATA_ROOT` and split with **`VAL_SPLIT`**.
- Use **Section 2b** to download from Kaggle into **`/content/nail-data`**, or point **`DATA_ROOT`** at your unzip folder.

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 8
EPOCHS = 60
# Used only when USE_OFFICIAL_SPLITS is False (single flat images/masks tree)
VAL_SPLIT = 0.15
USE_OFFICIAL_SPLITS = True
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# Default: where Section 2b places the Kaggle dataset (change if you unzip elsewhere)
DATA_ROOT = Path("/content/nail-data")

# Leave None unless you use manual dirs (flat layout)
IMAGE_DIR = None
MASK_DIR = None

## 2b. (Optional) Download from Kaggle

Dataset: **[nail-segmentation-dataset](https://www.kaggle.com/datasets/muhammadhammad261/nail-segmentation-dataset)** (`muhammadhammad261/nail-segmentation-dataset`).

1. On Kaggle: **Account → API → Create New API Token** → download `kaggle.json`.
2. In Colab, upload the file, then run the cell below (it extracts to **`/content/nail-data`** and sets **`DATA_ROOT`** if you re-run the config cell, or you can set `DATA_ROOT = Path("/content/nail-data")` only).

If you already unzipped the dataset manually, **skip** this cell and only set **`DATA_ROOT`** to that folder.

In [ ]:
# --- Optional: Kaggle download — run once per session ---
# !pip install -q kaggle
import os
import shutil
import subprocess
from pathlib import Path

KAGGLE_DATASET = "muhammadhammad261/nail-segmentation-dataset"
NAIL_DATA = Path("/content/nail-data")

try:
    from google.colab import files
except ImportError:
    files = None

if not (Path("/root/.kaggle/kaggle.json").exists()):
    if files is None:
        raise RuntimeError("Place /root/.kaggle/kaggle.json or run in Google Colab")
    print("Upload kaggle.json from Kaggle settings → API")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

NAIL_DATA.mkdir(parents=True, exist_ok=True)
subprocess.run(
    ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(NAIL_DATA), "--unzip"],
    check=True,
)

# Nested zip(s): unzip any remaining .zip inside
import zipfile
for z in list(NAIL_DATA.rglob("*.zip")):
    try:
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(z.parent)
        z.unlink(missing_ok=True)
    except zipfile.BadZipFile:
        pass

print("Top-level:", [p.name for p in sorted(NAIL_DATA.iterdir())][:20])
DATA_ROOT = NAIL_DATA
print("Set DATA_ROOT =", DATA_ROOT, "— re-run Section 2 config cell if needed")

## 3. Image–mask pairs

**Kaggle layout:** `train/images` + `train/masks`, **`valid/`** or **`val/`** + `images`/`masks`. Same stem per pair (e.g. `foo.jpg` ↔ `foo.png`). **`test/`** is optional for final eval.

If **`USE_OFFICIAL_SPLITS`** is `True`, train/val come from those folders. Otherwise the notebook **discovers** `images`/`masks` under **`DATA_ROOT`** and splits with **`VAL_SPLIT`**.

**Masks:** PNG grayscale **0 / 255** (decoded to float **[0, 1]**). **`NailSegmentationV1.csv`** is optional metadata — not used below.

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".webp"}

IMAGE_DIR_HINTS = (
    "images",
    "Images",
    "image",
    "JPEGImages",
    "imgs",
)
MASK_DIR_HINTS = (
    "masks",
    "Masks",
    "mask",
    "SegmentationClass",
    "labels",
    "Labels",
)


def count_pairs_between(img_dir: Path, mask_dir: Path) -> int:
    return len(list_pairs(img_dir, mask_dir))


def discover_pair_dirs(root: Path) -> Tuple[Path, Path]:
    """Find sibling images/ + masks/ folders under root (fallback layout)."""
    subs = [root]
    try:
        subs.extend([p for p in root.rglob("*") if p.is_dir()])
    except OSError:
        pass
    best = (None, None, -1)
    for parent in subs:
        dirs = [p for p in parent.iterdir() if p.is_dir()]
        for img_candidate in dirs:
            name_low = img_candidate.name.lower()
            if not any(
                h.lower() == name_low or h.lower() in name_low
                for h in IMAGE_DIR_HINTS
            ):
                continue
            for mask_candidate in dirs:
                if mask_candidate == img_candidate:
                    continue
                mn = mask_candidate.name.lower()
                if not any(
                    h.lower() == mn or h.lower() in mn for h in MASK_DIR_HINTS
                ):
                    continue
                n = count_pairs_between(img_candidate, mask_candidate)
                if n > best[2]:
                    best = (img_candidate, mask_candidate, n)
    if best[2] <= 0:
        raise RuntimeError(
            "Could not find matching images/masks under DATA_ROOT. "
            "Run: !find DATA_ROOT -type d | head -50"
        )
    print("Discovered IMAGE_DIR:", best[0])
    print("Discovered MASK_DIR:", best[1])
    print("Matching pairs:", best[2])
    return best[0], best[1]


def list_pairs(image_dir: Path, mask_dir: Path):
    pairs = []
    for p in sorted(image_dir.iterdir()):
        if p.suffix.lower() not in IMAGE_EXT:
            continue
        stem = p.stem
        m = None
        for ext in (".png", ".jpg", ".jpeg"):
            cand = mask_dir / f"{stem}{ext}"
            if cand.exists():
                m = cand
                break
        if m is None:
            cand_guess = mask_dir / (stem + "_mask.png")
            if cand_guess.exists():
                m = cand_guess
        if m is None:
            continue
        pairs.append((str(p), str(m)))
    return pairs


def resolve_validation_dir(root: Path) -> Optional[Path]:
    for name in ("valid", "val", "validation"):
        d = root / name
        if d.is_dir():
            return d
    return None


pairs_train: list
pairs_val: list
pairs_test: Optional[List[tuple]] = None

root = Path(DATA_ROOT)
train_root = root / "train"
val_root = resolve_validation_dir(root)

if USE_OFFICIAL_SPLITS and train_root.is_dir() and val_root is not None:
    ti, tm = train_root / "images", train_root / "masks"
    vi, vm = val_root / "images", val_root / "masks"
    assert ti.is_dir() and tm.is_dir(), f"Missing {ti} or {tm}"
    assert vi.is_dir() and vm.is_dir(), f"Missing {vi} or {vm}"
    pairs_train = list_pairs(ti, tm)
    pairs_val = list_pairs(vi, vm)
    print("Using official splits: train", len(pairs_train), "valid", len(pairs_val))
    test_root = root / "test"
    if test_root.is_dir():
        tei, tem = test_root / "images", test_root / "masks"
        if tei.is_dir() and tem.is_dir():
            pairs_test = list_pairs(tei, tem)
            print("test pairs:", len(pairs_test))
elif USE_OFFICIAL_SPLITS and train_root.is_dir() and val_root is None:
    raise RuntimeError(
        "Found train/ but no valid/ or val/ folder. Add validation split or set "
        "USE_OFFICIAL_SPLITS = False to use a single tree + VAL_SPLIT."
    )
elif IMAGE_DIR is not None and MASK_DIR is not None:
    pairs_all = list_pairs(Path(IMAGE_DIR), Path(MASK_DIR))
    n_val = max(1, int(len(pairs_all) * VAL_SPLIT))
    pairs_train = pairs_all[:-n_val]
    pairs_val = pairs_all[-n_val:]
    print("Manual dirs — split", len(pairs_train), "train", len(pairs_val), "val")
else:
    IMAGE_DIR, MASK_DIR = discover_pair_dirs(root)
    pairs_all = list_pairs(IMAGE_DIR, MASK_DIR)
    n_val = max(1, int(len(pairs_all) * VAL_SPLIT))
    pairs_train = pairs_all[:-n_val]
    pairs_val = pairs_all[-n_val:]
    print("Discovered flat layout — split", len(pairs_train), "train", len(pairs_val), "val")

assert len(pairs_train) > 0 and len(pairs_val) > 0, "Need train and val pairs"

## 4. `tf.data` pipeline (pure TF decode)

**Images:** JPG/RGB → float **[0, 1]**.

**Masks:** PNG — dataset uses **0 / 255** grayscale; we decode to float **[0, 1]** (`nail = 1`). Resize masks with **nearest**.

In [ ]:
def load_sample(img_path, mask_path):
    img_data = tf.io.read_file(img_path)
    img = tf.image.decode_image(img_data, channels=3, expand_animations=False)
    img = tf.image.convert_image_dtype(img, tf.float32)

    mask_raw = tf.io.read_file(mask_path)
    # Kaggle masks: grayscale PNG, 0 = background, 255 = nail
    mask_u8 = tf.image.decode_png(mask_raw, channels=1)
    mask = tf.cast(mask_u8, tf.float32) / 255.0
    mask = tf.clip_by_value(mask, 0.0, 1.0)

    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE], method="bilinear")
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.clip_by_value(mask, 0.0, 1.0)
    return img, mask

def augment(img, mask):
    seed = tf.random.uniform([2], maxval=10_000, dtype=tf.int32)
    img = tf.image.stateless_random_flip_left_right(img, seed=seed)
    mask = tf.image.stateless_random_flip_left_right(mask, seed=seed)
    return img, mask

def make_ds(paths_pairs, training: bool):
    ips = [a for a, _ in paths_pairs]
    mps = [b for _, b in paths_pairs]
    ds = tf.data.Dataset.from_tensor_slices((ips, mps))
    if training:
        ds = ds.shuffle(min(500, len(paths_pairs)), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_sample, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(pairs_train, training=True)
val_ds = make_ds(pairs_val, training=False)

test_ds = None
if pairs_test is not None and len(pairs_test) > 0:
    test_ds = make_ds(pairs_test, training=False)
    print("test_ds samples:", len(pairs_test))

print("train:", len(pairs_train), "val:", len(pairs_val))

## 5. Loss + batch IoU

`iou_metric` reduces **mean IoU per image** in the batch.

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [tf.shape(y_true)[0], -1])
    y_pred_f = tf.reshape(y_pred, [tf.shape(y_pred)[0], -1])
    inter = tf.reduce_sum(y_true_f * y_pred_f, axis=-1)
    denom = tf.reduce_sum(y_true_f, axis=-1) + tf.reduce_sum(y_pred_f, axis=-1)
    dice = (2.0 * inter + smooth) / (denom + smooth)
    return 1.0 - tf.reduce_mean(dice)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

def iou_metric(y_true, y_pred):
    pred_bin = tf.cast(y_pred > 0.5, tf.float32)
    inter = tf.reduce_sum(y_true * pred_bin, axis=[1, 2, 3])
    union = tf.reduce_sum(tf.maximum(y_true, pred_bin), axis=[1, 2, 3])
    iou = inter / (union + 1e-6)
    return tf.reduce_mean(iou)

## 6. Model — MobileNetV3-Small **U-Net with skip connections**

> **If you see** `InputLayer` **`has no attribute 'output_shape'`:** your Colab cell still has old code. Replace **`collect_stride_tensors`** with the version in this cell (do **not** use `layer.output_shape`). Re-fetch **`docs/nail_segmentation_colab_outline.ipynb`** from the repo if needed.

- Scan backbone layers and keep the **last tensor at each stride** `{32, 16, 8, 4}` (spatial grid `IMG_SIZE/stride`). Stride **4** may be absent on some TF builds — decoder falls back without that skip.
- **Decoder:** upsample with **`Conv2DTranspose`**, **concatenate** skip, **`Conv2D`** to mix channels. Repeat until **`IMG_SIZE`** is reached (typically **five** ×2 upsamples from stride **32**).
- Skips restore **fine boundaries** (cuticle, free edge) vs a plain decoder.

In [ ]:
def collect_stride_tensors(base: keras.Model, img_size: int):
    """Pick skip tensors by spatial stride. Never use ``layer.output_shape`` — in
    Keras 3, ``InputLayer`` has no ``output_shape`` attribute.
    """
    best = {}
    for layer in base.layers:
        try:
            out = layer.output
        except Exception:
            continue
        shp2 = tuple(keras.backend.int_shape(out))
        if len(shp2) != 4 or shp2[1] is None or shp2[2] is None:
            continue
        hi, wi = int(shp2[1]), int(shp2[2])
        if hi != wi or hi <= 0:
            continue
        if img_size % hi != 0:
            continue
        stride = img_size // hi
        if stride in (4, 8, 16, 32):
            best[stride] = out
    return best


def conv_block(x, filters: int, name=None):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu", name=name)(x)
    return x


def build_model(img_size: int) -> keras.Model:
    assert img_size % 32 == 0, "Use IMG_SIZE divisible by 32 (e.g. 256, 320)"
    inputs = keras.Input(shape=(img_size, img_size, 3))
    base = keras.applications.MobileNetV3Small(
        input_tensor=inputs,
        include_top=False,
        weights="imagenet",
        alpha=1.0,
        minimalistic=False,
        include_preprocessing=False,
    )
    skips = collect_stride_tensors(base, img_size)
    for s in (32, 16, 8):
        assert s in skips, (
            f"Need stride-{s} skip. Got strides {list(skips.keys())}. "
            "Try printing [(ly.name, keras.backend.int_shape(ly.output)) for ly in base.layers if getattr(ly, 'output', None) is not None]."
        )
    if 4 not in skips:
        print("Note: no stride-4 skip found — decoder continues without it.")

    print("Skip strides used:", sorted(skips.keys()))

    x = skips[32]

    x = layers.Conv2DTranspose(256, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x, skips[16]])
    x = conv_block(x, 256, "dec16")

    x = layers.Conv2DTranspose(128, 3, strides=2, padding="same", activation="relu")(x)
    x = layers.Concatenate()([x, skips[8]])
    x = conv_block(x, 128, "dec8")

    x = layers.Conv2DTranspose(64, 3, strides=2, padding="same", activation="relu")(x)
    if 4 in skips:
        x = layers.Concatenate()([x, skips[4]])
    x = conv_block(x, 64, "dec4")

    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same", activation="relu")(x)
    x = conv_block(x, 32, "up2")

    x = layers.Conv2DTranspose(16, 3, strides=2, padding="same", activation="relu")(x)
    x = conv_block(x, 16, "up1")

    out = layers.Conv2D(
        1, 1, activation="sigmoid", dtype="float32", name="nail_prob"
    )(x)
    return keras.Model(inputs, out, name="nail_seg_unet")


model = build_model(IMG_SIZE)
model.summary()

dummy = tf.zeros((1, IMG_SIZE, IMG_SIZE, 3))
assert model(dummy).shape == (1, IMG_SIZE, IMG_SIZE, 1)

### Optional: freeze backbone for early epochs

After `model.summary()`, find the MobileNet layer name (often contains `mobilenet`), then:

```python
enc = model.get_layer("mobilenetv3small")  # adjust to printed name
enc.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-4), loss=combined_loss, metrics=[iou_metric])
model.fit(train_ds, validation_data=val_ds, epochs=15)
enc.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss=combined_loss, metrics=[iou_metric])
model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
```

## 7. Train

**Run Sections 1 → 6 first.** Section 6 must finish successfully so **`model`** exists (`model = build_model(IMG_SIZE)`). If you restarted the runtime or skipped Section 6, you will see **`NameError: name 'model' is not defined`** — run Section 6 again.

In [ ]:
try:
    model
except NameError as e:
    raise RuntimeError(
        "Run Section 6 first (after 1–5): model = build_model(IMG_SIZE); "
        "If you restarted the kernel, re-run all cells above this one."
    ) from e

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[iou_metric],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        patience=10, restore_best_weights=True, monitor="val_iou_metric", mode="max"
    ),
    keras.callbacks.ModelCheckpoint(
        "best_nail.keras", save_best_only=True, monitor="val_iou_metric", mode="max"
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 8. Visual check

In [ ]:
def show_val_sample(idx=0):
    for imgs, masks in val_ds.take(1):
        pred = model.predict(imgs[:1], verbose=0)
        plt.figure(figsize=(12, 4))
        plt.subplot(1, 3, 1)
        plt.imshow(imgs[idx].numpy())
        plt.title("Image")
        plt.axis("off")
        plt.subplot(1, 3, 2)
        plt.imshow(masks[idx].numpy().squeeze(), vmin=0, vmax=1, cmap="gray")
        plt.title("GT mask")
        plt.axis("off")
        plt.subplot(1, 3, 3)
        plt.imshow(pred[idx].squeeze(), vmin=0, vmax=1, cmap="gray")
        plt.title("Predicted")
        plt.axis("off")
        plt.tight_layout()
        plt.show()
        break


def eval_test_set():
    if test_ds is None:
        print("No test_ds (no test/ split or empty).")
        return
    r = model.evaluate(test_ds, verbose=0)
    print("Test loss / metrics:", r)


# show_val_sample()
# eval_test_set()  # after training

## 9. Export SavedModel + TFLite (float32)

**Note:** On some Colab stacks, **`tf.saved_model.save(model, ...)`** raises **`TypeError: this __dict__ descriptor does not support '_DictWrapper'`** with Keras 3. This notebook uses **`model.export()`** or **`model.save(..., save_format="tf")`** first, and falls back to **`TFLiteConverter.from_keras_model(model)`** so you always get **`nail_seg.tflite`**.

Android expects **RGB float32 [0,1]**, **NHWC**, batch **1**. Output: nail probability **[0,1]** per pixel.

In [ ]:
import os

OUT_DIR = "saved_model_nail"

# 1) Try Keras export / save as SavedModel (avoid raw tf.saved_model.save bug on some TF+Keras3 builds)
saved_ok = False
try:
    model.export(OUT_DIR)
    saved_ok = os.path.isdir(OUT_DIR)
except (AttributeError, TypeError, ValueError) as e:
    print("model.export failed:", e)
if not saved_ok:
    try:
        model.save(OUT_DIR, save_format="tf")
        saved_ok = os.path.isdir(OUT_DIR)
    except Exception as e:
        print("SavedModel export skipped:", e)

# 2) Convert to TFLite
if saved_ok:
    converter = tf.lite.TFLiteConverter.from_saved_model(OUT_DIR)
else:
    print("Using from_keras_model (no SavedModel on disk)")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = []
tflite_bytes = converter.convert()
open("nail_seg.tflite", "wb").write(tflite_bytes)
print("Wrote nail_seg.tflite", len(tflite_bytes), "bytes")


**Fallback:** If disk export keeps failing, only **`from_keras_model`** runs — you still get **`nail_seg.tflite`**.

```python
converter = tf.lite.TFLiteConverter.from_keras_model(model)
```

## 10. Verify interpreter (same checks as Android)

In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inn = interp.get_input_details()[0]
out = interp.get_output_details()[0]
print("INPUT", inn["shape"], inn["dtype"])
print("OUTPUT", out["shape"], out["dtype"])

test_in = np.zeros((1, IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
interp.set_tensor(inn["index"], test_in)
interp.invoke()
print("out sample", interp.get_tensor(out["index"]).shape)

## 11. Ship to Android

1. Download **`nail_seg.tflite`** from Colab.
2. Place at **`app/src/main/assets/nail_seg.tflite`**.
3. Align **`NailSegmentationHelper`**: input `[1, H, W, 3]` float32, `/255` if you trained on `[0,1]` — **match preprocessing**.
4. Parse output tensor shape from **`getOutputTensor(0)`** (single channel vs multi-class).

**Debugging skips:** if `assert s in skips` fails, run:
```python
for ly in model.get_layer("mobilenet_v3_small").layers:
    print([(ly.name, keras.backend.int_shape(ly.output)) for ly in base.layers if getattr(ly, "output", None) is not None])
```
and adjust **`collect_stride_tensors`** to pick named layers by **`keras.backend.int_shape(layer.output)`** (e.g. `block_*_expand`).